In [1]:
ss1 = """# Task: 标的前瞻多空推演与交易决策生成

## 1. Global Definitions (知识查表层)

### 1.1 三层信息分类学与职责规范表
| 信息分类 | 核心系统职责 (`roles`) | 数据形态 (`data_form`) | 是否直接定多空 | 考核与评价规范 | 典型特征示例 |
| :--- | :--- | :---: | :---: | :--- | :--- |
| **市场观测与环境状态层 (Regime)** | `OBSERVATION`, `REGIME`, `RISK` | `NUMERIC` | **否** (仅限上下文) | 豁免单变量方向 IC 筛选；未做时序标准化;考核状态可分性、波动率区分与风险控制增益 | 均线排列、252日ATR波动率分位、估值水位(PE-TTM)、市场宽度均值 |
| **候选预测信号层 (Predictive)** | `PREDICTIVE` | `NUMERIC` | **是** (核心证据) | 必须接受样本外 IC/ICIR、方向准确率与 PnL 信用打分考核与动态淘汰,若没有根据逻辑和经验判断 | 主力资金流加速度(二阶导)、期现基差连续背离度、换手高位衰竭度、早盘跳空放量比 |
| **事件与文本信息层 (Textual)** | `OBSERVATION`, `RISK`, `PREDICTIVE` | `TEXT` | **辅助** (定性约束) | 严格 Point-in-time 审查；补充因果背景、政策定调与事件主体约束 | 货币政策定调、重大会议公报摘要、行业突发事件 (带 TF-ID) |

### 1.2 记忆主辅声明规则表 (仅在有召回记忆时生效)
| 记忆角色 (`usage`) | 数量约束 | 先验权重 ($UsageWeight_i$) | 职责与信用结算机制 |
| :--- | :---: | :---: | :--- |
| **primary (主记忆)** | **必须且只能有 1 条** | 0.70 (单条时为 1.0) | 与当前盘面微观博弈结构最相似的核心锚点，承担最大的收益/亏损权重 |
| **auxiliary (辅助记忆)** | 0 ~ N 条 | 0.30 / N (无辅助时为 0) | 补充次要维度的风险提示或反向边界校验 |

---

## 2. Navigation Topology (决策流转导航)

```mermaid
graph TD
    Start[接收时序数据与记忆库] --> Step1[步骤 1: 定性 Regime 环境底色与约束]
    Step1 --> Step2[步骤 2: 剖析 Predictive 时序演化与拐点]
    Step2 --> Step3{{记忆库是否包含经验?}}
    Step3 -- 否 (Zero-Shot 盲测态) --> Step4[步骤 4: 纯时序独立合成决策]
    Step3 -- 是 (RAG 增强态) --> Step3_1[步骤 3: 历史案例微观比对与先验校准]
    Step3_1 --> Step3_2[声明采信记忆、隐含方向与主辅关系]
    Step3_2 --> Step4
    Step4 --> Output[输出标准 JSON 决策协议]
```

---

## 3. Execution Logic (执行规则伪代码)

请严格参照以下 Python 业务逻辑进行思维推演：

```python
def generate_trading_prediction(regime_data, predictive_data, textual_events, retrieved_memories, holding_period):
    # 1. Regime 环境定性与策略模式选择
    regime_context = evaluate_regime_context(regime_data)
    # 若处于震荡/高波环境，顺势突破信号权重调低，反转/背离信号权重调高；若处于单边趋势，动量权重调高
    strategy_mode = 'MEAN_REVERSION' if regime_context.is_range_bound else 'TREND_FOLLOWING'
    
    # 2. Predictive 信号时序演化审查 (重点审查时序斜率、二阶导与背离)
    smart_money_acc = analyze_time_series_acceleration(predictive_data.get('main_flow_acceleration'))
    breadth_div = detect_divergence(predictive_data.get('breadth_div_intensity_pos'))
    basis_div = detect_divergence(predictive_data.get('basis_div_strength'))
    turnover_exhaustion = analyze_exhaustion(predictive_data.get('exhaustion_intensity'))
    
    # 3. 历史先验对齐 (Memory Alignment & Usage Declaration)
    used_memories_declaration = []
    if retrieved_memories and len(retrieved_memories) > 0:
        for mem in retrieved_memories:
            if is_microstructure_consistent(mem, predictive_data, regime_context):
                used_memories_declaration.append({{
                    'memory_id': mem.id,
                    'implied_direction': mem.suggested_direction,
                    'confidence': mem.match_confidence,
                    'usage': 'primary' if is_best_match(mem) else 'auxiliary'
                }})
        # 强制约束：若声明了记忆，必须且只能有 1 条 primary
        ensure_single_primary_memory(used_memories_declaration)
    
    # 4. 多空信号合成与置信度决断
    bull_signals, bear_signals = extract_evidences(predictive_data, textual_events, strategy_mode)
    
    if is_strong_bull_resonance(bull_signals, bear_signals, used_memories_declaration):
        direction = 'UP'
        confidence = calculate_calibrated_confidence(bull_signals, penalty=regime_context.risk_penalty)
    elif is_strong_bear_resonance(bear_signals, bull_signals, used_memories_declaration):
        direction = 'DOWN'
        confidence = calculate_calibrated_confidence(bear_signals, penalty=regime_context.risk_penalty)
    else:
        # 信号剧烈矛盾、处于诱多诱空陷阱或处于极端观望期
        direction = 'FLAT'
        confidence = 0.50
        
    # 5. 提取核心失效边界 (Invalidation Conditions)
    invalid_conditions = derive_invalidation_thresholds(predictive_data, direction)
    
    return {{
        'dominant_regime': regime_context.summary,
        'predict_direction': direction,
        'confidence': confidence,
        'used_features': [s.feature_name for s in (bull_signals if direction == 'UP' else bear_signals)],
        'predict_logic': format_causal_chains(bull_signals if direction == 'UP' else bear_signals),
        'used_memories': used_memories_declaration,
        'invalid_conditions': invalid_conditions
    }}
```

---

## 4. Data Container (数据容器层 - XML 隔离与注意力锚点)

<context_data>
<target_info>
- target_ticker: {ticker}
- holding_period: {holding_period}
</target_info>

<regime_features>
{regime_features_time_series}
</regime_features>

<predictive_signals>
{predictive_signals_time_series}
</predictive_signals>

<textual_events>
{textual_events_data}
</textual_events>

<retrieved_case_memories>
{retrieved_memories_block}
</retrieved_case_memories>
</context_data>

输出格式必须为:
{json_structure}

"""

In [2]:
print(ss1)

# Task: 标的前瞻多空推演与交易决策生成

## 1. Global Definitions (知识查表层)

### 1.1 三层信息分类学与职责规范表
| 信息分类 | 核心系统职责 (`roles`) | 数据形态 (`data_form`) | 是否直接定多空 | 考核与评价规范 | 典型特征示例 |
| :--- | :--- | :---: | :---: | :--- | :--- |
| **市场观测与环境状态层 (Regime)** | `OBSERVATION`, `REGIME`, `RISK` | `NUMERIC` | **否** (仅限上下文) | 豁免单变量方向 IC 筛选；未做时序标准化;考核状态可分性、波动率区分与风险控制增益 | 均线排列、252日ATR波动率分位、估值水位(PE-TTM)、市场宽度均值 |
| **候选预测信号层 (Predictive)** | `PREDICTIVE` | `NUMERIC` | **是** (核心证据) | 必须接受样本外 IC/ICIR、方向准确率与 PnL 信用打分考核与动态淘汰,若没有根据逻辑和经验判断 | 主力资金流加速度(二阶导)、期现基差连续背离度、换手高位衰竭度、早盘跳空放量比 |
| **事件与纯文本信息层 (Textual)** | `OBSERVATION`, `RISK`, `PREDICTIVE` | `TEXT` | **辅助** (定性约束) | 严格 Point-in-time 审查；补充因果背景、政策定调与事件主体约束 | 货币政策定调、重大会议公报摘要、行业突发事件 (带 TF-ID) |

### 1.2 记忆主辅声明规则表 (仅在有召回记忆时生效)
| 记忆角色 (`usage`) | 数量约束 | 先验权重 ($UsageWeight_i$) | 职责与信用结算机制 |
| :--- | :---: | :---: | :--- |
| **primary (主记忆)** | **必须且只能有 1 条** | 0.70 (单条时为 1.0) | 与当前盘面微观博弈结构最相似的核心锚点，承担最大的收益/亏损权重 |
| **auxiliary (辅助记忆)** | 0 ~ N 条 | 0.30 / N (无辅助时为 0) | 补充次要维度的风险提示

In [3]:
print(repr(ss1))

"# Task: 标的前瞻多空推演与交易决策生成\n\n## 1. Global Definitions (知识查表层)\n\n### 1.1 三层信息分类学与职责规范表\n| 信息分类 | 核心系统职责 (`roles`) | 数据形态 (`data_form`) | 是否直接定多空 | 考核与评价规范 | 典型特征示例 |\n| :--- | :--- | :---: | :---: | :--- | :--- |\n| **市场观测与环境状态层 (Regime)** | `OBSERVATION`, `REGIME`, `RISK` | `NUMERIC` | **否** (仅限上下文) | 豁免单变量方向 IC 筛选；未做时序标准化;考核状态可分性、波动率区分与风险控制增益 | 均线排列、252日ATR波动率分位、估值水位(PE-TTM)、市场宽度均值 |\n| **候选预测信号层 (Predictive)** | `PREDICTIVE` | `NUMERIC` | **是** (核心证据) | 必须接受样本外 IC/ICIR、方向准确率与 PnL 信用打分考核与动态淘汰,若没有根据逻辑和经验判断 | 主力资金流加速度(二阶导)、期现基差连续背离度、换手高位衰竭度、早盘跳空放量比 |\n| **事件与纯文本信息层 (Textual)** | `OBSERVATION`, `RISK`, `PREDICTIVE` | `TEXT` | **辅助** (定性约束) | 严格 Point-in-time 审查；补充因果背景、政策定调与事件主体约束 | 货币政策定调、重大会议公报摘要、行业突发事件 (带 TF-ID) |\n\n### 1.2 记忆主辅声明规则表 (仅在有召回记忆时生效)\n| 记忆角色 (`usage`) | 数量约束 | 先验权重 ($UsageWeight_i$) | 职责与信用结算机制 |\n| :--- | :---: | :---: | :--- |\n| **primary (主记忆)** | **必须且只能有 1 条** | 0.70 (单条时为 1.0) | 与当前盘面微观博弈结构最相似的核心锚点，承担最大的收益/亏损权重 |\n| **auxiliary (辅助记忆)** | 0 ~ N 条 | 0.30 / N (无辅助时为 